In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
# Import necessary libraries
import numpy as np
import pandas as pd
from sklearn.metrics import log_loss
from catboost import Pool, CatBoostClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.impute import SimpleImputer
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# Load the data
train_df = pd.read_csv('/kaggle/input/icr-identify-age-related-conditions/train.csv')
test_df = pd.read_csv('/kaggle/input/icr-identify-age-related-conditions/test.csv')
greeks_df = pd.read_csv('/kaggle/input/icr-identify-age-related-conditions/greeks.csv')
sample_submission_df = pd.read_csv('/kaggle/input/icr-identify-age-related-conditions/sample_submission.csv')


In [ ]:
train_df

In [ ]:
numerical_features = ['AB', 'AF', 'AH', 'AM', 'AR', 'AX', 'AY', 'AZ',
                      'BC', 'BD', 'BN', 'BP', 'BQ', 'BR', 'BZ',
                      'CB', 'CC', 'CD', 'CF', 'CH', 'CL', 'CR', 'CS', 'CU', 'CW',
                      'DA', 'DE', 'DF', 'DH', 'DI', 'DL', 'DN', 'DU', 'DV', 'DY',
                      'EB', 'EE', 'EG', 'EH', 'EL', 'EP', 'EU',
                      'FC', 'FD', 'FE', 'FI', 'FL', 'FR', 'FS',
                      'GB', 'GE', 'GF', 'GH', 'GI', 'GL']
categorical_features = ['EJ']
features = numerical_features + categorical_features

In [ ]:
train_df['EJ'] = train_df['EJ'].replace({'A': 0, 'B': 1})
test_df['EJ'] = test_df['EJ'].replace({'A': 0, 'B': 1})

In [ ]:
X=train_df.drop("Class",axis=1)
y=train_df["Class"]

X_test=test_df.copy()

In [ ]:
train_df["Class"].value_counts()

In [ ]:
pip install lightgbm

In [ ]:
pip install optuna

In [ ]:
# import optuna.integration.lightgbm as lgb
#import lightgbm as lgb
from imblearn.under_sampling import RandomUnderSampler
import numpy as np
from sklearn.metrics import log_loss
from sklearn.model_selection import KFold
import optuna.integration.lightgbm as lgb
import optuna
import pandas as pd
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
from imblearn.under_sampling import RandomUnderSampler

# params = {
#         #'boosting_type': 'DART',
#         'boosting_type': 'GBDT',
#         #'is_unbalance':'True',
#         #'objective': 'cross_entropy',
#         'objective':'binary',
#         'metric':'auc',
#        'return_cvbooster':'True',
# 'random_state': 42,}

params = {
    #'boosting_type': 'GBDT',
    #'boosting_type': 'DART',
    #'is_unbalance':'True',
    #'objective': 'cross_entropy',
    'objective':"binary",
    'metric':'binary_logloss',
    #'return_cvbooster':'True',
    'random_state': 42,
}


# Random seed average 20回  #4stは1回 5stは20回,6stは50回,7st1回 15st 20 16st 30 17st 50 10st 
for i in range(50):
    
    positive_count_train = y.value_counts()[1]
    #sampler = RandomUnderSampler(random_state=i, replacement=True)
    sampler = RandomUnderSampler(sampling_strategy={0: positive_count_train, 1: positive_count_train},random_state=i, replacement=True)
    X_re, y_re = sampler.fit_resample(X, y)
#     X_re, y_re = X, y

    # trainを学習データと検証データに分割
    (X_train, X_val , y_train , y_val) = train_test_split(X_re, y_re , test_size = 0.2,random_state=42)
    #LightGBM用データセットの作成
    #lgb_train = lgb.Dataset(X_train.drop("Id",axis=1), y_train,free_raw_data=False) #学習用
    lgb_eval = lgb.Dataset(X_val.drop("Id",axis=1), y_val,free_raw_data=False) #Boosting用
    lgb_train = lgb.Dataset(X_re.drop("Id",axis=1), y_re,free_raw_data=False) #学習用
    
    
    model = lgb.train(params, lgb_train, valid_sets=lgb_eval,
                      categorical_feature = categorical_features,
                      num_boost_round=1000,
                      early_stopping_rounds=20,
                      verbose_eval=10)
    
#     tuner=lgb.LightGBMTunerCV(params, lgb_train,verbose_eval=50,categorical_feature=categorical_features,
#                              return_cvbooster=True,early_stopping_rounds=20,num_boost_round=1000,
#                               folds=KFold(n_splits=4))
  
#     tuner.run()
#     # サーチしたパラメータの表示
#     best_params = tuner.best_params


# # 訓練で得た最良のモデル（Boosterオブジェクト）を取得する
#     best_model= tuner.get_best_booster()  
    pred = model.predict(X_test.drop("Id",axis=1),num_iteration=model.best_iteration)
#     pred= np.array(y_pred_proba_list).mean(axis=0)
    
    # Calculate and output log loss.
    #loss = log_loss(y_val, pred)
    #print(f"Log loss for model {i + 1}: {loss}")


#     #各予測結果を格納
#     # Store predictions.
    if i == 0:
        output = pd.DataFrame(pred, columns=['pred' + str(i + 1)])
        output2 = output
    else:
        output = pd.DataFrame(pred, columns=['pred' + str(i + 1)])
        output2 = pd.concat([output2, output], axis=1)
\
# #forの終わり
# # #各予測結果を平均
# # df_test["prob"] = output2.mean(axis='columns')
# # df_submit = df_test[['gid','prob']]
# # submit_cli(df_submit, "my_26_re_st_submit")
# output2.to_csv('test.csv', encording="cp932",index=False)


In [ ]:
# # import optuna.integration.lightgbm as lgb
# #import lightgbm as lgb
# from imblearn.under_sampling import RandomUnderSampler
# import numpy as np
# from sklearn.metrics import log_loss
# from sklearn.model_selection import KFold
# import optuna.integration.lightgbm as lgb
# import optuna
# import pandas as pd
# import numpy as np
# from sklearn.datasets import make_classification
# from sklearn.model_selection import train_test_split
# from sklearn.metrics import roc_auc_score
# from xgboost import XGBClassifier
# from sklearn.ensemble import RandomForestClassifier
# from imblearn.under_sampling import RandomUnderSampler

# # params = {
# #         #'boosting_type': 'DART',
# #         'boosting_type': 'GBDT',
# #         #'is_unbalance':'True',
# #         #'objective': 'cross_entropy',
# #         'objective':'binary',
# #         'metric':'auc',
# #        'return_cvbooster':'True',
# # 'random_state': 42,}

# params = {
#     'boosting_type': 'GBDT',
#     'is_unbalance':'True',
#     'objective': 'cross_entropy',
#     'metric':'binary_logloss',
#     'return_cvbooster':'True',
#     'random_state': 42,
# }


# # Random seed average 20回  #4stは1回 5stは20回,6stは50回,7st1回 15st 20 16st 30 17st 50 10st 
# for i in range(1):
    
#     positive_count_train = y.value_counts()[1]
#     #sampler = RandomUnderSampler(random_state=i, replacement=True)
#     #sampler = RandomUnderSampler(sampling_strategy={0: positive_count_train, 1: positive_count_train},random_state=i, replacement=True)
#     #X_re, y_re = sampler.fit_resample(X, y)
#     X_re, y_re = X, y

#     # trainを学習データと検証データに分割
#     (X_train, X_val , y_train , y_val) = train_test_split(X_re, y_re , test_size = 0.2,random_state=42)
#     #LightGBM用データセットの作成
#     #lgb_train = lgb.Dataset(X_train.drop("Id",axis=1), y_train,free_raw_data=False) #学習用
#     lgb_eval = lgb.Dataset(X_val.drop("Id",axis=1), y_val,free_raw_data=False) #Boosting用
#     lgb_train = lgb.Dataset(X_re.drop("Id",axis=1), y_re,free_raw_data=False) #学習用

#     tuner=lgb.LightGBMTunerCV(params, lgb_train,verbose_eval=50,categorical_feature=categorical_features,
#                              return_cvbooster=True,early_stopping_rounds=20,num_boost_round=1000,
#                               folds=KFold(n_splits=4))
  
#     tuner.run()
#     # サーチしたパラメータの表示
#     best_params = tuner.best_params


# # 訓練で得た最良のモデル（Boosterオブジェクト）を取得する
#     best_model= tuner.get_best_booster()  
#     y_pred_proba_list = best_model.predict(X_test.drop("Id",axis=1),num_iteration=best_model.best_iteration)
#     pred= np.array(y_pred_proba_list).mean(axis=0)
    
#     # Calculate and output log loss.
#     #loss = log_loss(y_val, pred)
#     #print(f"Log loss for model {i + 1}: {loss}")


# #     #各予測結果を格納
# #     # Store predictions.
# #     if i == 0:
# #         output = pd.DataFrame(pred, columns=['pred' + str(i + 1)])
# #         output2 = output
# #     else:
# #         output = pd.DataFrame(pred, columns=['pred' + str(i + 1)])
# #         output2 = pd.concat([output2, output], axis=1)

# # #forの終わり
# # # #各予測結果を平均
# # # df_test["prob"] = output2.mean(axis='columns')
# # # df_submit = df_test[['gid','prob']]
# # # submit_cli(df_submit, "my_26_re_st_submit")
# # output2.to_csv('test.csv', encording="cp932",index=False)


In [ ]:
 pred = output2.mean(axis='columns')

In [ ]:
submit=pd.DataFrame(test_df["Id"], columns=["Id"])

In [ ]:
submit["class_0"]=1-pred
submit["class_1"]=pred

In [ ]:
submit.to_csv('submission.csv',index=False)

In [ ]:
submit